In [1]:
import os
import chromadb
from google import genai
from pypdf import PdfReader
from dotenv import load_dotenv

In [34]:
print(load_dotenv())


True


In [35]:
api_key=os.getenv("GEMINI_API_KEY")

## 1.2 Initialize Gemini Client

In [4]:
client = genai.Client(api_key=api_key)

In [5]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello"
)

print(response.text)

Hello! How can I help you today?


In [9]:
import shutil

In [10]:
shutil.copy(
    r"C:\Users\User\Downloads\thinkpython2.pdf",
    "documents/"
)

'documents/thinkpython2.pdf'

In [11]:
shutil.copy(r"C:\Users\User\Downloads\thinkstats2.pdf","documents/")

'documents/thinkstats2.pdf'

In [12]:
shutil.copy(r"C:\Users\User\Downloads\thinkcomplexity2.pdf","documents/")

'documents/thinkcomplexity2.pdf'

In [13]:
print(os.listdir("documents"))

['thinkcomplexity2.pdf', 'thinkpython2.pdf', 'thinkstats2.pdf']


In [14]:
files = os.listdir("documents")

#1.3 data processing and chunking

In [16]:
pdf_files = []

In [17]:

for file in files:
    if file.endswith(".pdf"):
        pdf_files.append(file)


In [18]:
print(pdf_files)

['thinkcomplexity2.pdf', 'thinkpython2.pdf', 'thinkstats2.pdf']


In [19]:
file_path=[]

In [20]:
for file in pdf_files:
    file_path.append(os.path.join("documents",file))

In [21]:
all_pages=[]

In [22]:
for file in file_path:
    pdf = PdfReader(file)

    for i, page in enumerate(pdf.pages):
        text=page.extract_text()
        page_number=(i+1)
        all_pages.append({
    "pdf_name": file,
    "text": text,
    "page": page_number
})

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


In [23]:
print(len(all_pages))

744


In [24]:
print(all_pages[0]["pdf_name"])

documents\thinkcomplexity2.pdf


In [25]:
print(all_pages[0]["text"][:500])

Think Complexity
Version 2.6.4
The third edition ofThink Complexityis in progress at
allendowney.github.io/ThinkComplexity


In [26]:
for item in all_pages:
    print(item["pdf_name"])

documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
documents\thinkcomplexity2.pdf
document

In [27]:
for file in file_path:
    count=0
    for item in all_pages:
        if item["pdf_name"]==file:
            count+=1
    print(file,count)

documents\thinkcomplexity2.pdf 236
documents\thinkpython2.pdf 244
documents\thinkstats2.pdf 264


In [28]:
chunks=[]

In [29]:
for page in all_pages:
    extracted_text = page["text"]

    if extracted_text.strip() != "":
        sentence_text = extracted_text.replace("?", ".")
        sentences = sentence_text.split(".")

        sentences = [
            s.strip() for s in sentences
            if s.strip() != ""
        ]

        current_chunk = ""

        for sentence in sentences:

            # Handle sentences longer than 500 characters
            if len(sentence) > 500:

                # Save the current chunk first
                if current_chunk.strip():
                    chunks.append({
                        "chunk": current_chunk.strip(),
                        "pdf_name": page["pdf_name"],
                        "page_number": page["page"]
                    })

                current_chunk = ""

                # Split the long sentence into smaller pieces
                for start in range(0, len(sentence), 500):
                    piece = sentence[start:start+500]

                    chunks.append({
                        "chunk": piece,
                        "pdf_name": page["pdf_name"],
                        "page_number": page["page"]
                    })

            # Handle normal sentences
            elif len(current_chunk) + len(sentence) + 1 <= 500:
                current_chunk += sentence + " "

            else:
                chunks.append({
                    "chunk": current_chunk.strip(),
                    "pdf_name": page["pdf_name"],
                    "page_number": page["page"]
                })

                current_chunk = sentence + " "

        # Save the remaining text
        if current_chunk.strip():
            chunks.append({
                "chunk": current_chunk.strip(),
                "pdf_name": page["pdf_name"],
                "page_number": page["page"]
            })

In [30]:
print(len(chunks))

2770


In [31]:
print(chunks[0])

{'chunk': 'Think Complexity\nVersion 2 6 4\nThe third edition ofThink Complexityis in progress at\nallendowney github io/ThinkComplexity', 'pdf_name': 'documents\\thinkcomplexity2.pdf', 'page_number': 1}


In [32]:
max(len(chunk["chunk"]) for chunk in chunks)

500

In [33]:
print(all(chunk["chunk"].strip() != "" for chunk in chunks))

True


In [36]:
from sentence_transformers import SentenceTransformer

c:\Users\User\.conda\envs\pythonLab\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
embedding_model=SentenceTransformer("all-MiniLM-L6-v2")

In [38]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [39]:
test_embedding=embedding_model.encode(chunks[0]["chunk"])
print(test_embedding.shape)

(384,)


In [40]:
new_embeddings = embedding_model.encode(
    [chunk["chunk"] for chunk in chunks],
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches: 100%|██████████| 87/87 [00:58<00:00,  1.48it/s]


In [41]:
print(new_embeddings.shape)
print(len(new_embeddings))

(2770, 384)
2770


In [42]:
print(new_embeddings[0])

[ 3.63670103e-02 -5.59083968e-02 -3.38375643e-02 -4.26722988e-02
 -3.07688746e-03 -4.95451391e-02  3.34229507e-02  1.04861595e-02
  4.32711327e-03  3.23980786e-02 -6.09667413e-02  2.84149386e-02
  3.46757136e-02  2.78609246e-03  7.32890703e-03  6.63398579e-02
 -2.95421593e-02 -2.31356695e-02  5.91583841e-04 -5.88159226e-02
  6.08308241e-02 -6.73478544e-02 -1.90717932e-02 -1.29802451e-02
 -1.42289372e-02  1.08020213e-02 -8.07467625e-02 -4.57273908e-02
  8.18885788e-02  1.57415830e-02  5.07406238e-03  7.69658610e-02
  4.59538028e-02 -1.05665147e-03  9.79188159e-02  7.26759881e-02
  3.59125994e-03 -5.76999085e-03  1.99173056e-02  6.68934174e-03
 -6.31309375e-02  1.77538414e-02  3.35085280e-02 -4.93393168e-02
 -1.15882726e-02  8.05776790e-02  2.68441415e-03 -5.90498075e-02
 -8.42501894e-02 -4.93395701e-02 -1.29318550e-01 -1.50262807e-02
 -8.14724341e-02 -2.32046153e-02  6.43494949e-02 -3.56921053e-04
  2.49458458e-02  1.71027780e-02 -4.60688919e-02 -9.00947824e-02
 -3.86077613e-02 -4.45730

# 1.7 Store Embeddings in ChromaDB

In [43]:
chroma_client = chromadb.Client()

In [44]:
pdf_documents = chroma_client.create_collection(
    name="pdf_documents"
)

In [45]:
ids = []
documents = []
metadatas = []

In [46]:
for i, chunk in enumerate(chunks):
    ids.append(f"chunk_{i}")
    documents.append(chunk["chunk"])

    metadatas.append({
        "pdf_filename": chunk["pdf_name"],
        "page": chunk["page_number"]
    })

In [47]:
print(len(ids))
print(len(documents))
print(len(metadatas))
print(metadatas[0])

2770
2770
2770
{'pdf_filename': 'documents\\thinkcomplexity2.pdf', 'page': 1}


In [48]:
pdf_documents.add(
    ids=ids,
    documents=documents,
    embeddings=new_embeddings.tolist(),
    metadatas=metadatas
)

In [49]:
print(pdf_documents.count())

2770


## 1.8.1 Prepare the User Question

In [50]:
question = "What does concatenation mean in programming?"

## 1.8.2 Generate Question Embedding

In [51]:
question_embedding = embedding_model.encode(question)

## 1.8.3 Search for Relevant Chunks

In [52]:
results = pdf_documents.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=5
)

In [53]:
retrieved_documents = results["documents"][0]

print(retrieved_documents)

['On the other hand, there is a significant way in which string concatenation and repe-\ntition are different from integer addition and multiplication Can you think of a property\nthat addition has that string concatenation does not 2 7 Comments\nAs programs get bigger and more complicated, they get more difficult to read Formal\nlanguages are dense, and it is often difficult to look at a piece of code and figure out what\nit is doing, or why', 'In this example,print_twicewas called bycat_twice, andcat_twicewas called\nby__main__, which is a special name for the topmost frame When you create a variable\noutside of any function, it belongs to__main__ Each parameter refers to the same value as its corresponding argument So,part1has the\nsame value asline1,part2has the same value asline2, andbrucehas the same value as\ncat', "Chapter 2\nVariables, expressions and\nstatements\nOne of the most powerful features of a programming language is the ability to manipulate\nvariables A variable is 

## 1.8.4 Extract Sources

In [54]:
sources = results["metadatas"][0]

for source in sources:
    print(source["pdf_filename"], source["page"])

documents\thinkpython2.pdf 34
documents\thinkpython2.pdf 45
documents\thinkpython2.pdf 31
documents\thinkpython2.pdf 44
documents\thinkpython2.pdf 31


In [55]:
print("Retrieved chunks:", len(retrieved_documents))
print("Sources:", sources)

Retrieved chunks: 5
Sources: [{'page': 34, 'pdf_filename': 'documents\\thinkpython2.pdf'}, {'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 45}, {'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 31}, {'page': 44, 'pdf_filename': 'documents\\thinkpython2.pdf'}, {'page': 31, 'pdf_filename': 'documents\\thinkpython2.pdf'}]


## 1.9.1 Generate Answer

In [56]:
def generate_answer(question, retrieved_documents):

    context = "\n\n".join(retrieved_documents)

    prompt = f"""
    You are an assistant that answers questions about PDF documents.

    Instructions:
    - Answer using only the provided context.
    - You may make reasonable conclusions based on the context.
    - Do not invent facts.
    - If the answer is not available, say:
      "The answer is not mentioned in the provided context."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [57]:
answer = generate_answer(question, retrieved_documents)

print(answer)

The answer is not mentioned in the provided context.


In [58]:
for i, doc in enumerate(retrieved_documents):
    print(f"Chunk {i+1}:")
    print(doc)
    print("-" * 50)

Chunk 1:
On the other hand, there is a significant way in which string concatenation and repe-
tition are different from integer addition and multiplication Can you think of a property
that addition has that string concatenation does not 2 7 Comments
As programs get bigger and more complicated, they get more difficult to read Formal
languages are dense, and it is often difficult to look at a piece of code and figure out what
it is doing, or why
--------------------------------------------------
Chunk 2:
In this example,print_twicewas called bycat_twice, andcat_twicewas called
by__main__, which is a special name for the topmost frame When you create a variable
outside of any function, it belongs to__main__ Each parameter refers to the same value as its corresponding argument So,part1has the
same value asline1,part2has the same value asline2, andbrucehas the same value as
cat
--------------------------------------------------
Chunk 3:
Chapter 2
Variables, expressions and
statements
One o

In [59]:
for i, chunk in enumerate(chunks):
    if "This function takes two arguments" in chunk["chunk"]:
        print("Chunk index:", i)
        print(chunk["chunk"])

Chunk index: 964
It doesn’t matter what the value was called back home (in
the caller); here inprint_twice, we call everybodybruce 3 8 Variables and parameters are local
When you create a variable inside a function, it islocal, which means that it only exists
inside the function For example:
def cat_twice(part1, part2):
cat = part1 + part2
print_twice(cat)
This function takes two arguments, concatenates them, and prints the result twice


## 1.10.1 Keyword Search

In [60]:
question = "What does concatenation mean in programming?"

question = question.lower().strip()
keywords = question.split()

print(keywords)

['what', 'does', 'concatenation', 'mean', 'in', 'programming?']


In [63]:
keywords = ["concatenation", "concatenates"]
matching_chunks = []

for chunk in chunks:
    chunk_text = chunk["chunk"].lower()

    for keyword in keywords:
        if keyword in chunk_text:
            matching_chunks.append(chunk)
            break

In [64]:
print("Matching chunks:", len(matching_chunks))

Matching chunks: 13


In [65]:
for chunk in matching_chunks:
    if "This function takes two arguments" in chunk["chunk"]:
        print(chunk)

{'chunk': 'It doesn’t matter what the value was called back home (in\nthe caller); here inprint_twice, we call everybodybruce 3 8 Variables and parameters are local\nWhen you create a variable inside a function, it islocal, which means that it only exists\ninside the function For example:\ndef cat_twice(part1, part2):\ncat = part1 + part2\nprint_twice(cat)\nThis function takes two arguments, concatenates them, and prints the result twice', 'pdf_name': 'documents\\thinkpython2.pdf', 'page_number': 44}


In [66]:
combined_chunks = []

In [67]:
for doc in retrieved_documents:
    combined_chunks.append(doc)

In [68]:
print(len(combined_chunks))

5


In [70]:
for chunk in matching_chunks:
    if chunk["chunk"] not in combined_chunks:
        combined_chunks.append(chunk["chunk"])

In [71]:
print(len(combined_chunks))

17


In [72]:
for doc in combined_chunks:
    if "This function takes two arguments" in doc:
        print(doc)

It doesn’t matter what the value was called back home (in
the caller); here inprint_twice, we call everybodybruce 3 8 Variables and parameters are local
When you create a variable inside a function, it islocal, which means that it only exists
inside the function For example:
def cat_twice(part1, part2):
cat = part1 + part2
print_twice(cat)
This function takes two arguments, concatenates them, and prints the result twice


In [73]:
answer = generate_answer(question, combined_chunks)

print(answer)

Based on the provided context, concatenation means joining strings (or lists) end-to-end. For example, string concatenation joins strings by linking them end-to-end, and the `+` operator can be used to concatenate strings or lists.


# 1.11 Complete RAG Pipeline

In [74]:

def ask_pdf(question):

    # 1. Generate question embedding
    question_embedding = embedding_model.encode(question)

    # 2. Semantic search
    results = pdf_documents.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=5
    )

    retrieved_documents = results["documents"][0]
    semantic_sources = results["metadatas"][0]

    # 3. Prepare semantic results with their sources
    combined_results = []

    for doc, source in zip(retrieved_documents, semantic_sources):
        combined_results.append({
            "text": doc,
            "source": source
        })

    # 4. Prepare meaningful keywords
    stop_words = {
        "what", "does", "mean", "in", "the", "is",
        "a", "an", "how", "why", "to", "of", "and",
        "for", "can", "explain"
    }

    keywords = [
        word.strip(".,?!;:")
        for word in question.lower().split()
        if word.strip(".,?!;:") not in stop_words
        and len(word.strip(".,?!;:")) > 2
    ]

    # 5. Keyword search
    for chunk in chunks:
        chunk_text = chunk["chunk"].lower()

        if any(keyword in chunk_text for keyword in keywords):

            already_exists = any(
                item["text"] == chunk["chunk"]
                for item in combined_results
            )

            if not already_exists:
                combined_results.append({
                    "text": chunk["chunk"],
                    "source": {
                        "pdf_filename": chunk["pdf_name"],
                        "page": chunk["page_number"]
                    }
                })

    # 6. Prepare context for Gemini
    final_documents = [
        item["text"] for item in combined_results
    ]

    # 7. Generate answer
    answer = generate_answer(question, final_documents)

    # 8. Extract unique sources
    unique_sources = []

    for item in combined_results:
        source = item["source"]

        if source not in unique_sources:
            unique_sources.append(source)

    return answer, unique_sources, final_documents


# 1.12 Testing the RAG Pipeline

In [75]:
question = "What does concatenation mean in programming?"

answer, sources, retrieved_documents = ask_pdf(question)

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print(source)

print("\nNumber of retrieved chunks:", len(retrieved_documents))

Answer:
Based on the provided context, string concatenation means joining strings by linking them end-to-end.

Sources:
{'page': 34, 'pdf_filename': 'documents\\thinkpython2.pdf'}
{'page': 45, 'pdf_filename': 'documents\\thinkpython2.pdf'}
{'page': 31, 'pdf_filename': 'documents\\thinkpython2.pdf'}
{'page': 44, 'pdf_filename': 'documents\\thinkpython2.pdf'}
{'pdf_filename': 'documents\\thinkcomplexity2.pdf', 'page': 13}
{'pdf_filename': 'documents\\thinkcomplexity2.pdf', 'page': 14}
{'pdf_filename': 'documents\\thinkcomplexity2.pdf', 'page': 58}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 5}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 8}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 23}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 24}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 26}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 28}
{'pdf_filename': 'documents\\thinkpython2.pdf', 'page': 29}
{'pdf_filename': 'documents\\t

# 1.13 Evaluation and Testing

In [76]:
evaluation_questions = [
    "What is a variable in Python?",
    "What is a histogram?",
    "What is correlation?",
    "What is complexity science?",
    "What is the capital of Japan?"
]

In [77]:
evaluation_results = []

for question in evaluation_questions:
    answer, sources, retrieved_documents = ask_pdf(question)

    evaluation_results.append({
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_documents": retrieved_documents
    })

    print("Question:", question)
    print("Answer:", answer)
    print("-" * 60)

Question: What is a variable in Python?
Answer: Based on the provided context, a variable is a name that refers to a value.
------------------------------------------------------------
Question: What is a histogram?
Answer: Based on the provided context, a histogram is:

- A mapping from values to frequencies, or a graph that shows this mapping.
- A graph that shows the frequency of each value in a variable's distribution (where "frequency" means the number of times the value appears).
- A collection of counters (or frequencies).
------------------------------------------------------------
Question: What is correlation?
Answer: Based on the provided context, correlation is a statistic intended to quantify the strength of the relationship between two variables.
------------------------------------------------------------
Question: What is complexity science?
Answer: Based on the provided context, complexity science is an interdisciplinary field—at the intersection of mathematics, comput